# 📷 Google Photos Takeout 雲端自動化增量整理工坊 (本地 SSD 分波防斷線版)

本筆記本專為 **Google One 2TB** 雲端生態設計，直接結合 `14_Google-Photos-Takeout-Organizer` 官方核心引擎。

### 核心亮點：
1. **100% 本地高速 SSD 解壓與分波處理**：所有 ZIP 先高速載入至 Colab 本地 `/content` SSD 解壓分析，**絕不將數萬個暫存小檔解壓回 Google Drive**，徹底根絕 `Errno 107: Transport endpoint is not connected` FUSE 斷線問題！
2. **智慧批次過濾**：自動鎖定最新一波 Takeout 匯出，並自動排除重複下載的 `(1).zip` 副本與檢查缺卷。
3. **即時百分比進度顯示**：解壓、配對、歸檔、驗證每一步都有清晰進度條與檔案計數。
4. **全域 Sidecar 雜湊配對 & 五層日期保底**：秒級解析數萬筆 JSON 時間戳記，跨分波自動保留 JSON 候選池。
5. **SHA-256 增量去重與安全清理**：所有分波 100% 驗證通過後，才安全清理來源 ZIP 釋放雲端空間。

## 步驟 1：掛載 Google 雲端硬碟（含 FUSE 斷線自動自癒）
點擊左側播放鍵 ▶️，依提示授權存取 Google Drive：

In [ ]:
from google.colab import drive
import os, sys, shutil, glob, time, re, subprocess
from pathlib import Path

# 1. 鎖定台灣標準時區 (Asia/Taipei, UTC+8)
!ln -snf /usr/share/zoneinfo/Asia/Taipei /etc/localtime && echo "Asia/Taipei" > /etc/timezone
os.environ['TZ'] = 'Asia/Taipei'
try:
    time.tzset()
except AttributeError:
    pass

# 2. 若前次執行造成 /content/drive FUSE 斷線 (Errno 107)，先強制卸載再重新掛載
try:
    _ = os.listdir('/content/drive/MyDrive')
except OSError:
    subprocess.run(['fusermount', '-uz', '/content/drive'], stderr=subprocess.DEVNULL)
    subprocess.run(['umount', '-l', '/content/drive'], stderr=subprocess.DEVNULL)

drive.mount('/content/drive', force_remount=True)
print('✓ Google Drive 掛載成功且時區已鎖定為台灣標準時間 (Asia/Taipei)！')

## 步驟 2：安裝 14 號專案核心引擎與相依套件

In [ ]:
# 先安裝影像解碼相依套件，再以 --no-deps 極速安裝最新核心引擎（免下載桌面版 PySide6）
!pip install -q pillow pillow-heif
!pip install -q --upgrade --force-reinstall --no-deps git+https://github.com/lianghao02/14_Google-Photos-Takeout-Organizer.git
print('✓ 核心整理引擎最新版本安裝完成！')


## 步驟 3：設定路徑、智慧鎖定最新匯出批次並規劃本地 SSD 分波

In [ ]:
from collections import defaultdict

DRIVE_ROOT = Path('/content/drive/MyDrive')
TAKEOUT_DIR = DRIVE_ROOT / 'Takeout'
OUTPUT_DIR = DRIVE_ROOT / 'Organized_Photos'

# 預設僅處理最新一波 Takeout 匯出，並自動略過重複的 (1).zip 副本
ONLY_LATEST_BATCH = True
SKIP_DUPLICATE_COPIES = True

TAKEOUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 若雲端硬碟殘留舊版解壓在 Drive 上的 Takeout_Work，主動清除以釋放雲端空間並避免 FUSE 負載
legacy_drive_work = DRIVE_ROOT / 'Takeout_Work'
if legacy_drive_work.exists():
    print('🧹 偵測到舊版留在 Google Drive 上的 Takeout_Work 暫存區，正在清理釋放空間...')
    shutil.rmtree(legacy_drive_work, ignore_errors=True)
    print('✓ 舊版雲端暫存區已清理完成！')

all_zip_files = sorted(list(TAKEOUT_DIR.glob('*.zip')))
zip_pat = re.compile(r'^takeout-(\d{8}T\d{6}Z)-(?:(\d+)-)?(\d{3})(?:\s*\((\d+)\))?\.zip$', re.IGNORECASE)

batches = defaultdict(list)
other_zips = []
for z in all_zip_files:
    m = zip_pat.match(z.name)
    if m:
        batch_id, part_grp, vol_idx, copy_num = m.groups()
        batches[batch_id].append((z, part_grp, int(vol_idx), copy_num))
    else:
        other_zips.append(z)

skipped_old_zips = []
skipped_copy_zips = []
zip_files = []

if batches:
    sorted_batch_ids = sorted(batches.keys())
    target_batch_ids = [sorted_batch_ids[-1]] if ONLY_LATEST_BATCH else sorted_batch_ids
    if ONLY_LATEST_BATCH and len(sorted_batch_ids) > 1:
        for old_id in sorted_batch_ids[:-1]:
            skipped_old_zips.extend(item[0] for item in batches[old_id])
        print(f'ℹ️ 偵測到多個匯出批次 {sorted_batch_ids}，已自動鎖定最新批次：{target_batch_ids[0]}（略過舊批次 {len(skipped_old_zips)} 檔）')

    for b_id in target_batch_ids:
        items = batches[b_id]
        by_vol = defaultdict(list)
        for item in items:
            by_vol[(item[1], item[2])].append(item)
        vol_numbers = sorted(k[1] for k in by_vol.keys() if k[0] is not None)
        if vol_numbers:
            missing_vols = sorted(set(range(1, max(vol_numbers) + 1)) - set(vol_numbers))
            if missing_vols:
                print(f'⚠️ 注意：批次 {b_id} 缺少分卷編號：{missing_vols}')
        for key in sorted(by_vol.keys(), key=lambda x: (x[0] or '', x[1])):
            candidates = sorted(by_vol[key], key=lambda x: (x[3] is not None, int(x[3] or 0)))
            if SKIP_DUPLICATE_COPIES and len(candidates) > 1:
                zip_files.append(candidates[0][0])
                skipped_copy_zips.extend(c[0] for c in candidates[1:])
            else:
                zip_files.extend(c[0] for c in candidates)

zip_files.extend(other_zips)
if skipped_copy_zips:
    saved_gb = sum(z.stat().st_size for z in skipped_copy_zips) / (1024 ** 3)
    print(f'ℹ️ 已自動排除同批次重複 (1).zip 副本：{len(skipped_copy_zips)} 檔（省下 {saved_gb:.1f} GB 重複解壓）')

total_size_gb = sum(z.stat().st_size for z in zip_files) / (1024 ** 3)

# 強制一律使用 Colab 本地高速 SSD (/content/Takeout_Work)，徹底避免在 Google Drive FUSE 解壓小檔導致 Errno 107 斷線
WORK_DIR = Path('/content/Takeout_Work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

colab_total, colab_used, colab_free = shutil.disk_usage('/content')
colab_free_gb = colab_free / (1024 ** 3)
# 每波次安全容量上限：保留 20GB 緩衝，每波最多處理約 25~40GB 的 ZIP，處理完即釋放本地空間
wave_budget_gb = max(10.0, min(35.0, (colab_free_gb - 20.0) * 0.65))

waves = []
current_wave = []
current_wave_gb = 0.0
for z in zip_files:
    z_gb = z.stat().st_size / (1024 ** 3)
    if current_wave and (current_wave_gb + z_gb > wave_budget_gb):
        waves.append(current_wave)
        current_wave = [z]
        current_wave_gb = z_gb
    else:
        current_wave.append(z)
        current_wave_gb += z_gb
if current_wave:
    waves.append(current_wave)

print(f'相簿目標目錄：{OUTPUT_DIR}')
print(f'本次實際處理 Takeout 壓縮檔：{len(zip_files)} 卷 (總計約 {total_size_gb:.1f} GB)')
print(f'Colab 本地 SSD 可用空間：{colab_free_gb:.1f} GB (單波上限：{wave_budget_gb:.1f} GB，共分為 {len(waves)} 波執行)')
print(f'工作暫存區：⚡ {WORK_DIR} (100% 本地高速 SSD 分波模式，零 FUSE 解壓斷線風險)')

for idx, z in enumerate(zip_files[:5], 1):
    size_mb = z.stat().st_size / (1024 * 1024)
    print(f'  [{idx:03d}] {z.name} ({size_mb:.1f} MB)')
if len(zip_files) > 5:
    print(f'  ... 略過中間 {len(zip_files)-5} 個檔案 ...')
    last_z = zip_files[-1]
    print(f'  [{len(zip_files):03d}] {last_z.name} ({last_z.stat().st_size / (1024*1024):.1f} MB)')

if not zip_files:
    print('\n⚠️ 目前 Takeout 目錄下沒有 ZIP 檔案。若 Google 尚未完成打包，請稍候再試。')

## 步驟 4：執行本地 SSD 分波分析、去重與無損匯出 (含 FUSE 自動重連與即時進度)

In [ ]:
import sys, time, subprocess
for mod in list(sys.modules.keys()):
    if 'google_photos_takeout_organizer' in mod:
        del sys.modules[mod]

if not zip_files:
    raise RuntimeError('沒有待處理的 ZIP 檔案，流程終止。')

from google_photos_takeout_organizer.service import analyze
from google_photos_takeout_organizer.exporter import export
from google_photos_takeout_organizer.verifier import verify

def ensure_drive_connected():
    """確保 /content/drive 連線正常，若遇瞬斷自動重新掛載"""
    try:
        _ = os.listdir('/content/drive/MyDrive')
    except OSError:
        print('\n🔄 偵測到 Google Drive 通道瞬斷，正在自動重新連線...')
        subprocess.run(['fusermount', '-uz', '/content/drive'], stderr=subprocess.DEVNULL)
        subprocess.run(['umount', '-l', '/content/drive'], stderr=subprocess.DEVNULL)
        drive.mount('/content/drive', force_remount=True)
        print('✓ Google Drive 自動重新連線成功！')

last_update = [0]
def console_progress(stage, current, total, message):
    now = time.time()
    pct = (current / total * 100) if total else 0
    if now - last_update[0] > 0.3 or current == total or current == 1:
        last_update[0] = now
        stage_labels = {
            'ANALYZE_EXTRACT': '📦 本地SSD解壓',
            'ANALYZE': '🔎 分析配對中',
            'EXPORT': '📁 歸檔寫入中',
            'VERIFY': '🛡️ 完整性驗證',
        }
        prefix = stage_labels.get(stage, '⏳ 處理中')
        sys.stdout.write(f'\r{prefix} [{pct:5.1f}%] ({current}/{total}) - {message[:50]}')
        sys.stdout.flush()
        if current == total:
            sys.stdout.write('\n')

total_verified = 0
total_failed = 0
all_waves_passed = True

for wave_idx, wave_zips in enumerate(waves, 1):
    wave_gb = sum(z.stat().st_size for z in wave_zips) / (1024 ** 3)
    print(f'\n====================================================')
    print(f'🚀 開始執行第 {wave_idx}/{len(waves)} 波 ({len(wave_zips)} 卷 ZIP，共 {wave_gb:.1f} GB)')
    print(f'====================================================')

    ensure_drive_connected()
    manifest = analyze(wave_zips, WORK_DIR, progress_callback=console_progress)
    manifest_path = WORK_DIR / 'manifest.json'
    print(f'✓ 第 {wave_idx}/{len(waves)} 波分析完成：共 {manifest["summary"]["total_media"]} 筆媒體 (配對 JSON: {manifest["summary"]["json_matched"]})')

    ensure_drive_connected()
    export_result = export(manifest_path, OUTPUT_DIR, progress_callback=console_progress, export_sidecars=False)

    ensure_drive_connected()
    w_verify = verify(manifest_path, OUTPUT_DIR, progress_callback=console_progress)
    total_verified += w_verify.get('verified', 0)
    total_failed += w_verify.get('failed', 0)
    if w_verify.get('result') != 'PASS':
        all_waves_passed = False
        print(f'⚠️ 第 {wave_idx} 波有部分檔案未通過驗證：{w_verify}')
    else:
        print(f'✓ 第 {wave_idx}/{len(waves)} 波驗證 PASS (累計已驗證 {total_verified} 筆)')

    # 釋放本地 SSD 已匯出的媒體檔案，僅保留極小的 .json 供後續波次跨卷配對
    extracted_root = WORK_DIR / 'extracted'
    if extracted_root.exists():
        for sub in extracted_root.iterdir():
            if sub.is_dir() and sub.name.startswith('archive_'):
                target_pool = extracted_root / f'json_pool_w{wave_idx}_{sub.name}'
                if target_pool.exists():
                    shutil.rmtree(target_pool, ignore_errors=True)
                for p in list(sub.rglob('*')):
                    if p.is_file() and p.suffix.lower() != '.json':
                        p.unlink(missing_ok=True)
                sub.replace(target_pool)

verify_result = {'result': 'PASS' if (all_waves_passed and total_failed == 0) else 'FAIL', 'verified': total_verified, 'failed': total_failed}
print('\n====================================================')
print('🏁 全波次最終驗證結果：', verify_result)
print('====================================================')

if verify_result.get('result') == 'PASS':
    print('\n🎉 所有相片與影片 100% 完整無誤寫入 Organized_Photos！')
else:
    print('\n⚠️ 部分檔案校驗未通過，請查看上方訊息。')

## 步驟 5：安全清理 Takeout 壓縮檔與暫存區 (釋放雲端硬碟空間)
**注意**：僅在全波次校驗 100% 通過 (`result == 'PASS'`) 時才會刪除原始 ZIP。

In [ ]:
# 設定是否刪除已處理的 ZIP 檔案與重複/舊批次 ZIP (預設為 True 釋放空間)
DELETE_SOURCE_ZIPS = True
DELETE_SKIPPED_OLD_AND_COPY_ZIPS = True

is_verified_ok = verify_result.get('result') == 'PASS' or verify_result.get('status') == 'OK'
if is_verified_ok:
    ensure_drive_connected()
    if DELETE_SOURCE_ZIPS:
        targets = list(zip_files)
        if DELETE_SKIPPED_OLD_AND_COPY_ZIPS:
            targets.extend(skipped_copy_zips)
            targets.extend(skipped_old_zips)
        for z in targets:
            try:
                if z.exists():
                    z.unlink()
                    print(f'✓ 已安全刪除封存檔：{z.name}')
            except Exception as e:
                print(f'刪除失敗 {z.name}: {e}')

    # 清理本地 SSD 暫存區
    extracted_dir = WORK_DIR / 'extracted'
    if extracted_dir.exists():
        shutil.rmtree(extracted_dir, ignore_errors=True)
        print(f'✓ 已清理本地暫存目錄：{extracted_dir}')

    print('\n✨ 任務圓滿完成！雲端硬碟空間已釋放，相片已妥善歸檔至 Organized_Photos/')
else:
    print('❌ 因驗證未完全通過，已嚴格保留所有原始 ZIP 檔案與暫存區，確保資料絕對安全！')